In [1]:
import os
import sys
import subprocess

import h5py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm

# Install e2cnn if missing
try:
    import e2cnn
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "e2cnn"
    ])

from e2cnn import gspaces
from e2cnn import nn as enn

# ============================================================
# 1. Device and Hyperparameter Configuration
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

BATCH_SIZE   = 50
EPOCHS       = 15
LR           = 1e-4
WEIGHT_DECAY = 1e-5

# ============================================================
# 2. Read H5 Data and Build Dataset & DataLoader
# ============================================================
with h5py.File("atom_train_data.h5", "r") as f:
    images = f["images"][:]
    labels = f["labels"][:]

# Validate image shape and binary labels
if images.ndim != 3 or images.shape[1:] != (7, 7):
    raise ValueError(
        f"Expected images with shape [N, 7, 7], got {images.shape}"
    )

if labels.ndim != 1 or len(labels) != len(images):
    raise ValueError(
        f"Expected labels with shape [{len(images)}], got {labels.shape}"
    )

if not ((labels == 0) | (labels == 1)).all():
    raise ValueError("Labels must contain only 0 and 1.")

# Convert images to [N, 1, 7, 7]
X = torch.tensor(images, dtype=torch.float32).unsqueeze(1)
y = torch.tensor(labels, dtype=torch.long)

full_dataset = TensorDataset(X, y)

# Use the same reproducible 80/20 split as the ordinary CNN
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

if train_size < 2 or test_size < 1:
    raise ValueError("Not enough samples for training and evaluation.")

train_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(
    f"Total samples: {len(full_dataset)} | "
    f"Train: {train_size} | Test: {test_size}"
)

# ============================================================
# 3. C4-Equivariant Model Architecture
# ============================================================
class C4AtomCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # Define rotations by multiples of 90 degrees
        self.r2_act = gspaces.Rot2dOnR2(N=4)

        # Input: one grayscale channel
        self.input_type = enn.FieldType(
            self.r2_act,
            [self.r2_act.trivial_repr],
        )

        # Each regular field contains four orientation channels
        type1 = enn.FieldType(
            self.r2_act,
            4 * [self.r2_act.regular_repr],
        )
        type2 = enn.FieldType(
            self.r2_act,
            8 * [self.r2_act.regular_repr],
        )
        type3 = enn.FieldType(
            self.r2_act,
            4 * [self.r2_act.regular_repr],
        )

        # [B, 1, 7, 7] -> [B, 16, 5, 5]
        self.block1 = enn.SequentialModule(
            enn.R2Conv(
                self.input_type, type1,
                kernel_size=3, padding=0, bias=False,
            ),
            enn.InnerBatchNorm(type1),
            enn.ReLU(type1, inplace=True),
        )

        # [B, 16, 5, 5] -> [B, 32, 3, 3]
        self.block2 = enn.SequentialModule(
            enn.R2Conv(
                type1, type2,
                kernel_size=3, padding=0, bias=False,
            ),
            enn.InnerBatchNorm(type2),
            enn.ReLU(type2, inplace=True),
        )

        # [B, 32, 3, 3] -> [B, 16, 1, 1]
        self.block3 = enn.SequentialModule(
            enn.R2Conv(
                type2, type3,
                kernel_size=3, padding=0, bias=False,
            ),
            enn.InnerBatchNorm(type3),
            enn.ReLU(type3, inplace=True),
        )

        # Pool four orientations within each field:
        # [B, 16, 1, 1] -> [B, 4, 1, 1]
        self.gpool = enn.GroupPooling(type3)

        # Map invariant features to two class scores
        self.fc = nn.Linear(self.gpool.out_type.size, num_classes)

    def forward(self, x):
        x = enn.GeometricTensor(x, self.input_type)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.gpool(x).tensor
        x = x.flatten(start_dim=1)

        return self.fc(x)


model = C4AtomCNN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
loss_fn = nn.CrossEntropyLoss()

num_params = sum(p.numel() for p in model.parameters())
print(f"C4ENN model parameters: {num_params:,}")

# ============================================================
# 4. Two Separate Evaluation Functions
# ============================================================
def evaluate_normal(loader):
    """Evaluate the original, unrotated images."""
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for x, t in loader:
            x, t = x.to(device), t.to(device)

            pred = model(x).argmax(dim=1)

            correct += (pred == t).sum().item()
            total += t.size(0)

    return 100.0 * correct / total


def evaluate_rotated(loader):
    """Evaluate all 90, 180, and 270 degree versions."""
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for x, t in loader:
            x, t = x.to(device), t.to(device)

            for rot_k in [1, 2, 3]:
                x_rot = torch.rot90(
                    x, k=rot_k, dims=[2, 3]
                )

                pred = model(x_rot).argmax(dim=1)

                correct += (pred == t).sum().item()
                total += t.size(0)

    return 100.0 * correct / total

# ============================================================
# 5. Training Loop
# ============================================================
print("\n--- Training C4ENN on Cold Atom Dataset ---")

for epoch in tqdm(range(EPOCHS)):
    model.train()

    for x, t in train_loader:
        x, t = x.to(device), t.to(device)

        optimizer.zero_grad()
        loss = loss_fn(model(x), t)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        normal_acc = evaluate_normal(test_loader)
        rotated_acc = evaluate_rotated(test_loader)

        print(
            f"Epoch {epoch + 1:2d} | "
            f"Normal Test Acc: {normal_acc:6.2f}% | "
            f"Rotated Test Acc: {rotated_acc:6.2f}%"
        )

# ============================================================
# 6. Final Evaluation
# ============================================================
final_normal_acc = evaluate_normal(test_loader)
final_rotated_acc = evaluate_rotated(test_loader)

print("\n" + "=" * 60)
print("Final C4ENN Evaluation Summary")
print("=" * 60)
print(f"  Normal Test Acc:  {final_normal_acc:6.2f}%")
print(f"  Rotated Test Acc: {final_rotated_acc:6.2f}%")
print("=" * 60)

# ============================================================
# 7. Save Model Weights
# ============================================================
os.makedirs("models", exist_ok=True)
ckpt_path = "models/c4_atom_cnn.pth"
torch.save(model.state_dict(), ckpt_path)
print(f"Checkpoint saved to {ckpt_path}")

Using device: cpu
Total samples: 13216 | Train: 10572 | Test: 2644


/usr/local/lib/python3.13/dist-packages/e2cnn/nn/modules/r2_conv/basisexpansion_singleblock.py:80: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:37.)
  full_mask[mask] = norms.to(torch.uint8)


C4ENN model parameters: 1,602

--- Training C4ENN on Cold Atom Dataset ---


  7%|▋         | 1/15 [00:02<00:28,  2.05s/it]

Epoch  1 | Normal Test Acc:  52.42% | Rotated Test Acc:  52.42%


...

 67%|██████▋   | 10/15 [00:17<00:09,  1.86s/it]

Epoch 10 | Normal Test Acc:  99.89% | Rotated Test Acc:  99.89%


100%|██████████| 15/15 [00:25<00:00,  1.72s/it]

Epoch 15 | Normal Test Acc:  99.92% | Rotated Test Acc:  99.92%



Final C4ENN Evaluation Summary
  Normal Test Acc:   99.92%
  Rotated Test Acc:  99.92%
Checkpoint saved to models/c4_atom_cnn.pth
